# GenAI-Traces: Prompt Management & A/B Testing

This notebook demonstrates:
- Prompt versioning and registry
- Template compilation
- Version diffing and rollback
- A/B testing experiments
- Statistical significance testing

In [1]:
import sys
sys.path.insert(0, '..')

# Clean up any previous test files
import os
for f in ['./test_prompts.json', './test_experiments.json']:
    if os.path.exists(f):
        os.remove(f)

## 1. Prompt Registry

In [2]:
from genai_traces.prompt_management import PromptRegistry, PromptVersion

registry = PromptRegistry(storage_path="./test_prompts.json")

# Save prompt versions
v1 = registry.save(
    name="summarize",
    template="Summarize the following text:\n\n{{text}}",
    version="1.0.0",
    labels=["staging"],
    metadata={"author": "alice", "model": "gpt-4o"}
)

v2 = registry.save(
    name="summarize",
    template="Summarize the following text in {{max_words}} words or less:\n\n{{text}}",
    version="1.1.0",
    labels=["production"],
    metadata={"author": "bob", "model": "gpt-4o"}
)

print("Saved prompts:")
print(f"  v1.0.0 hash: {v1.template_hash}")
print(f"  v1.1.0 hash: {v2.template_hash}")
print(f"\nVersions: {registry.list_versions('summarize')}")

Saved prompts:
  v1.0.0 hash: 2d659c962836
  v1.1.0 hash: a68d8243cb02

Versions: ['1.0.0', '1.1.0']


In [3]:
# Fetch prompts
print("Fetching prompts:")
print("=" * 60)

# By version
prompt_v1 = registry.get("summarize", version="1.0.0")
print(f"\nBy version (1.0.0):")
print(f"  Template: {prompt_v1.template[:50]}...")

# By label
prompt_prod = registry.get("summarize", label="production")
print(f"\nBy label (production):")
print(f"  Version: {prompt_prod.version}")
print(f"  Template: {prompt_prod.template[:50]}...")

# Latest (default)
prompt_latest = registry.get("summarize")
print(f"\nLatest:")
print(f"  Version: {prompt_latest.version}")

Fetching prompts:

By version (1.0.0):
  Template: Summarize the following text:

{{text}}...

By label (production):
  Version: 1.1.0
  Template: Summarize the following text in {{max_words}} word...

Latest:
  Version: 1.1.0


## 2. Template Compilation

In [4]:
# Compile template with variables
prompt = registry.get("summarize", label="production")

compiled = prompt.compile(
    max_words=100,
    text="This is a long document about artificial intelligence and its applications in healthcare..."
)

print("Compiled prompt:")
print("=" * 60)
print(compiled)

Compiled prompt:
Summarize the following text in 100 words or less:

This is a long document about artificial intelligence and its applications in healthcare...


## 3. Version Diffing

In [5]:
# Diff between versions
diff = registry.diff("summarize", "1.0.0", "1.1.0")

print("Diff between v1.0.0 and v1.1.0:")
print("=" * 60)
print(diff)

Diff between v1.0.0 and v1.1.0:
--- summarize@1.0.0
+++ summarize@1.1.0
@@ -1,3 +1,3 @@
-Summarize the following text:
+Summarize the following text in {{max_words}} words or less:
 
 {{text}}


## 4. Rollback

In [6]:
# Rollback production to v1.0.0
print("Before rollback:")
prod = registry.get("summarize", label="production")
print(f"  Production version: {prod.version}")

registry.rollback("summarize", to_version="1.0.0", label="production")

print("\nAfter rollback:")
prod = registry.get("summarize", label="production")
print(f"  Production version: {prod.version}")

Before rollback:
  Production version: 1.1.0

After rollback:
  Production version: 1.0.0


## 5. A/B Testing

In [7]:
from genai_traces.prompt_management import ABTestManager, Experiment

ab = ABTestManager(storage_path="./test_experiments.json")

# Create an experiment
experiment = ab.create_experiment(
    experiment_id="summarize_style_test",
    variants=[
        {"id": "control", "prompt_name": "summarize", "version": "1.0.0", "weight": 0.5},
        {"id": "concise", "prompt_name": "summarize", "version": "1.1.0", "weight": 0.5},
    ]
)

print("Created experiment:")
print(f"  ID: {experiment.experiment_id}")
print(f"  Status: {experiment.status}")
print(f"  Variants: {[v.id for v in experiment.variants]}")

Created experiment:
  ID: summarize_style_test
  Status: active
  Variants: ['control', 'concise']


In [8]:
# Test consistent user assignment
print("\nUser assignment (consistent):")
print("=" * 60)

for user_id in ["user_1", "user_2", "user_3", "user_1", "user_2"]:
    variant = ab.get_variant("summarize_style_test", user_id=user_id)
    print(f"  {user_id} -> {variant.id}")

print("\nNote: Same user always gets same variant!")


User assignment (consistent):
  user_1 -> control
  user_2 -> concise
  user_3 -> control
  user_1 -> control
  user_2 -> concise

Note: Same user always gets same variant!


In [9]:
# Simulate recording results
import random

print("\nSimulating experiment results...")

# Simulate 100 users
for i in range(100):
    user_id = f"user_{i}"
    variant = ab.get_variant("summarize_style_test", user_id=user_id)
    
    # Simulate metrics (concise variant performs slightly better)
    if variant.id == "concise":
        quality = random.gauss(0.82, 0.1)
        latency = random.gauss(450, 50)
    else:
        quality = random.gauss(0.78, 0.1)
        latency = random.gauss(500, 50)
    
    ab.record_result("summarize_style_test", variant.id, "quality", quality)
    ab.record_result("summarize_style_test", variant.id, "latency_ms", latency)

print("Recorded 100 experiment results.")


Simulating experiment results...
Recorded 100 experiment results.


In [10]:
# Get results summary
summary = ab.get_results_summary("summarize_style_test")

print("\nExperiment Results Summary:")
print("=" * 60)

for variant_id, metrics in summary.items():
    print(f"\n{variant_id}:")
    for metric, stats in metrics.items():
        print(f"  {metric}: mean={stats['mean']:.4f}, stdev={stats.get('stdev', 0):.4f}, n={stats['n']}")


Experiment Results Summary:

control:
  quality: mean=0.7715, stdev=0.0885, n=57
  latency_ms: mean=496.8805, stdev=50.3762, n=57

concise:
  quality: mean=0.8299, stdev=0.1111, n=43
  latency_ms: mean=444.1204, stdev=44.7350, n=43


In [11]:
# Check statistical significance
try:
    significance = ab.check_significance(
        experiment_id="summarize_style_test",
        metric="quality",
        variant_a="control",
        variant_b="concise",
        alpha=0.05
    )
    
    print("\nStatistical Significance Test (quality):")
    print("=" * 60)
    print(f"  t-statistic: {significance['t_statistic']:.4f}")
    print(f"  p-value: {significance['p_value']:.4f}")
    print(f"  Significant at α=0.05: {significance['significant']}")
    print(f"  Sample sizes: control={significance['n_a']}, concise={significance['n_b']}")
    
    if significance['significant']:
        print("\n✅ The difference is statistically significant!")
    else:
        print("\n⚠️ The difference is not statistically significant yet.")
except ImportError:
    print("Note: scipy required for significance testing. Install with: pip install scipy")


Statistical Significance Test (quality):
  t-statistic: -2.9275
  p-value: 0.0042
  Significant at α=0.05: True
  Sample sizes: control=57, concise=43

✅ The difference is statistically significant!


## 6. Integrated A/B Testing with Tracing

In [12]:
from genai_traces import init_tracer
from genai_traces.exporters import ConsoleExporter
from genai_traces.core.types import SpanType

tracer = init_tracer(
    service_name="ab-test-demo",
    exporters=[ConsoleExporter(pretty=True)],
)

def run_ab_tested_llm(user_id: str, text: str):
    """Run LLM with A/B tested prompt."""
    # Activate experiment (sets context)
    variant = ab.activate("summarize_style_test", user_id=user_id)
    
    # Get the prompt for this variant
    prompt_version = registry.get(variant.prompt_name, version=variant.version)
    
    with tracer.start_as_current_span("ab_tested_summarize", SpanType.LLM) as span:
        # Experiment context is automatically attached
        span.set_attribute("prompt.name", variant.prompt_name)
        span.set_attribute("prompt.version", variant.version)
        
        # Compile and use prompt
        if variant.id == "concise":
            compiled = prompt_version.compile(max_words=50, text=text)
        else:
            compiled = prompt_version.compile(text=text)
        
        span.set_attribute("llm.prompt", compiled)
        span.set_attribute("llm.completion", f"Summary for variant {variant.id}...")
        
        return variant.id

# Test
print("Running A/B tested LLM call:")
print("=" * 60)
variant_used = run_ab_tested_llm("test_user_123", "This is a long document...")
print(f"\nVariant used: {variant_used}")

Running A/B tested LLM call:
[SPAN] ab_tested_summarize (llm) - ok
{
  "trace_id": "f796ec7c4e9f4b81a4b85d4ca31d59d9",
  "span_id": "5a137ad06f455569",
  "parent_span_id": null,
  "root_span_id": "5a137ad06f455569",
  "name": "ab_tested_summarize",
  "span_type": "llm",
  "start_time": "2026-04-04T19:58:38.899804",
  "end_time": "2026-04-04T19:58:38.899804",
  "duration_ms": 0.0,
  "status": "ok",
  "status_message": null,
  "attributes": {
    "service.name": "ab-test-demo",
    "service.environment": "development",
    "service.version": "0.0.0",
    "experiment.id": "summarize_style_test",
    "experiment.variant": "control",
    "prompt.name": "summarize",
    "prompt.version": "1.0.0",
    "llm.prompt": "Summarize the following text:\n\nThis is a long document...",
    "llm.completion": "Summary for variant control..."
  },
  "events": [],
  "links": [],
  "context": {},
  "prompt_name": null,
  "prompt_version": null,
  "experiment_id": "summarize_style_test",
  "variant_id": "co

In [13]:
# Cleanup
import os
for f in ['./test_prompts.json', './test_experiments.json']:
    if os.path.exists(f):
        os.remove(f)
print("Cleaned up test files.")

Cleaned up test files.


## Summary

This notebook demonstrated:
- ✅ Prompt versioning and registry
- ✅ Template compilation with variables
- ✅ Version diffing
- ✅ Label-based deployment and rollback
- ✅ A/B experiment creation
- ✅ Consistent user assignment
- ✅ Result recording and summary
- ✅ Statistical significance testing
- ✅ Integrated A/B testing with tracing